# Universal Precision Runtime (UPR) — Notebook 02
## Level 2 (Layer Outputs) & Level 3 (Logit & Generation Comparison)

---

### Objective
1. **Level 2 Evaluation:** Register forward hooks across all Transformer block layers to compare activation hidden states (L2 Error, Cosine Similarity, Max Difference).
2. **Level 3 Evaluation:** Compare raw output logits (MAE, RMSE, Cosine Similarity, KL Divergence).
3. **Generation Evaluation:** Generate text sequences and calculate Top-1 / Top-5 token agreement percentages.
4. Memory-optimized sequential evaluation to guarantee zero OOM kernel crashes.

### Step 1: Environment Setup & Automatic Package Deployment
Set up Hugging Face authentication, mount Google Drive, and import `upr` package.

In [1]:
import os
import sys
import gc
import torch

# Set Hugging Face Token
HF_TOKEN = "YOUR_HF_TOKEN_HERE"
os.environ["HF_TOKEN"] = HF_TOKEN

# Mount Google Drive if in Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted successfully.')
    DRIVE_DIR = '/content/drive/MyDrive/UniversalPrecisionRuntime'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    os.chdir(DRIVE_DIR)
except ImportError:
    print('Running in local environment.')

WORK_DIR = os.getcwd()
print(f"Active Working Directory: {WORK_DIR}")
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

# Automatic Bootstrap: Ensure package files exist
os.makedirs("upr", exist_ok=True)

with open("upr/__init__.py", "w", encoding="utf-8") as f:
    f.write('''from .bit_ops import (
    float16_to_uint16_numpy,
    uint16_to_float16_torch,
    extract_bit_plane_np,
    pack_bit_plane,
    unpack_bit_plane,
    reconstruct_tensor,
)
from .converter import convert_to_bitplanes
from .loader import BitPlaneModel
from .metrics import compute_weight_metrics

__version__ = "0.1.0"
__all__ = [
    "float16_to_uint16_numpy",
    "uint16_to_float16_torch",
    "extract_bit_plane_np",
    "pack_bit_plane",
    "unpack_bit_plane",
    "reconstruct_tensor",
    "convert_to_bitplanes",
    "BitPlaneModel",
    "compute_weight_metrics",
]
''')

with open("upr/bit_ops.py", "w", encoding="utf-8") as f:
    f.write('''import torch
import numpy as np
from typing import Tuple, Dict, Optional, Union

def float16_to_uint16_numpy(tensor: torch.Tensor) -> np.ndarray:
    np_f16 = tensor.detach().cpu().to(torch.float16).numpy()
    return np_f16.view(np.uint16)

def uint16_to_float16_torch(np_uint16: np.ndarray, device: Union[str, torch.device] = 'cpu') -> torch.Tensor:
    np_f16 = np_uint16.view(np.float16)
    return torch.from_numpy(np_f16).to(device)

def extract_bit_plane_np(uint16_arr: np.ndarray, bit_index: int) -> np.ndarray:
    assert 0 <= bit_index <= 15, f"bit_index must be between 0 and 15, got {bit_index}"
    return ((uint16_arr >> bit_index) & 1).astype(np.uint8)

def pack_bit_plane(bit_arr: np.ndarray) -> bytes:
    flat = bit_arr.ravel()
    packed = np.packbits(flat, bitorder='big')
    return packed.tobytes()

def unpack_bit_plane(packed_bytes: bytes, num_elements: int, shape: Optional[Tuple[int, ...]] = None) -> np.ndarray:
    packed_np = np.frombuffer(packed_bytes, dtype=np.uint8)
    unpacked = np.unpackbits(packed_np, bitorder='big')[:num_elements]
    if shape is not None:
        unpacked = unpacked.reshape(shape)
    return unpacked.astype(np.uint8)

def reconstruct_tensor(planes_dict: Dict[int, bytes], selected_bits: int, original_shape: Tuple[int, ...], device: Union[str, torch.device] = 'cpu') -> torch.Tensor:
    assert 1 <= selected_bits <= 16, f"selected_bits must be between 1 and 16, got {selected_bits}"
    num_elements = int(np.prod(original_shape)) if len(original_shape) > 0 else 1
    accum = np.zeros(num_elements, dtype=np.uint32)
    start_bit = 15
    end_bit = 16 - selected_bits
    for b in range(start_bit, end_bit - 1, -1):
        if b in planes_dict:
            bits = unpack_bit_plane(planes_dict[b], num_elements)
            accum |= (bits.astype(np.uint32) << b)
    uint16_arr = accum.astype(np.uint16).reshape(original_shape)
    return uint16_to_float16_torch(uint16_arr, device=device)
''')

with open("upr/converter.py", "w", encoding="utf-8") as f:
    f.write('''import os
import json
import torch
import numpy as np
from typing import Union, Optional
from tqdm import tqdm
from transformers import AutoModelForCausalLM
from .bit_ops import float16_to_uint16_numpy, extract_bit_plane_np, pack_bit_plane

def convert_to_bitplanes(model_or_path: Union[str, torch.nn.Module], output_directory: str, torch_dtype: torch.dtype = torch.float16) -> str:
    os.makedirs(output_directory, exist_ok=True)
    tensors_dir = os.path.join(output_directory, "tensors")
    os.makedirs(tensors_dir, exist_ok=True)
    if isinstance(model_or_path, str):
        model_name = model_or_path
        print(f"Loading Hugging Face model from: {model_name}")
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch_dtype, low_cpu_mem_usage=True)
    else:
        model_name = getattr(model_or_path, "name_or_path", "custom_model")
        model = model_or_path
    state_dict = model.state_dict()
    metadata = {"model_name_or_path": model_name, "num_tensors": len(state_dict), "tensors": {}}
    print(f"Converting {len(state_dict)} tensors to bit-plane format in '{output_directory}'...")
    for idx, (tensor_name, tensor) in enumerate(tqdm(state_dict.items(), desc="BitPlane Conversion")):
        tensor_folder_name = f"tensor_{idx}"
        tensor_folder_path = os.path.join(tensors_dir, tensor_folder_name)
        os.makedirs(tensor_folder_path, exist_ok=True)
        original_shape = list(tensor.shape)
        dtype_str = str(tensor.dtype).replace("torch.", "")
        uint16_arr = float16_to_uint16_numpy(tensor)
        planes_meta = {}
        for bit_idx in range(16):
            plane_filename = f"plane{bit_idx}.bin"
            plane_path = os.path.join(tensor_folder_path, plane_filename)
            bit_arr = extract_bit_plane_np(uint16_arr, bit_idx)
            packed_bytes = pack_bit_plane(bit_arr)
            with open(plane_path, "wb") as f:
                f.write(packed_bytes)
            planes_meta[str(bit_idx)] = f"tensors/{tensor_folder_name}/{plane_filename}"
        metadata["tensors"][tensor_name] = {"shape": original_shape, "dtype": dtype_str, "numel": int(tensor.numel()), "folder": f"tensors/{tensor_folder_name}", "planes": planes_meta}
    metadata_path = os.path.join(output_directory, "metadata.json")
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)
    print(f"Successfully converted model to BitPlane format at: {output_directory}")
    return output_directory
''')

with open("upr/loader.py", "w", encoding="utf-8") as f:
    f.write('''import os\nimport json\nimport gc\nimport torch\nfrom typing import Optional, Union, Dict, Any\nfrom tqdm import tqdm\nfrom transformers import AutoModelForCausalLM, AutoConfig\nfrom .bit_ops import reconstruct_tensor\n\nclass BitPlaneModel:\n    @classmethod\n    def load_reconstructed_state_dict(cls, bitplane_directory: str, bits: int = 16, device: Union[str, torch.device] = 'cpu') -> Dict[str, torch.Tensor]:\n        assert 1 <= bits <= 16, f"bits must be between 1 and 16, got {bits}"\n        metadata_path = os.path.join(bitplane_directory, "metadata.json")\n        if not os.path.exists(metadata_path):\n            raise FileNotFoundError(f"metadata.json not found in '{bitplane_directory}'")\n        with open(metadata_path, "r", encoding="utf-8") as f:\n            metadata = json.load(f)\n        reconstructed_state_dict = {}\n        tensors_meta = metadata["tensors"]\n        start_bit = 15\n        end_bit = 16 - bits\n        for tensor_name, info in tqdm(tensors_meta.items(), desc=f"Reconstructing ({bits}-bit)"):\n            original_shape = tuple(info["shape"])\n            planes_dict = {}\n            for b in range(start_bit, end_bit - 1, -1):\n                plane_rel_path = info["planes"][str(b)]\n                plane_full_path = os.path.join(bitplane_directory, plane_rel_path)\n                if os.path.exists(plane_full_path):\n                    with open(plane_full_path, "rb") as pf:\n                        planes_dict[b] = pf.read()\n            recon_tensor = reconstruct_tensor(planes_dict=planes_dict, selected_bits=bits, original_shape=original_shape, device=device)\n            del planes_dict\n            reconstructed_state_dict[tensor_name] = recon_tensor\n        return reconstructed_state_dict\n\n    @classmethod\n    def from_pretrained(cls, bitplane_directory: str, bits: int = 16, base_model_id: Optional[str] = None, device_map: Optional[Union[str, Dict[str, Any]]] = None, torch_dtype: torch.dtype = torch.float16, **kwargs) -> torch.nn.Module:\n        metadata_path = os.path.join(bitplane_directory, "metadata.json")\n        with open(metadata_path, "r", encoding="utf-8") as f:\n            metadata = json.load(f)\n        model_name = base_model_id or metadata.get("model_name_or_path")\n        print(f"Instantiating model base architecture '{model_name}' for precision bits={bits}...")\n        config = AutoConfig.from_pretrained(model_name)\n        model = AutoModelForCausalLM.from_config(config, torch_dtype=torch_dtype)\n        state_dict = cls.load_reconstructed_state_dict(bitplane_directory=bitplane_directory, bits=bits, device='cpu')\n        model.load_state_dict(state_dict, strict=True)\n        del state_dict\n        gc.collect()\n        if device_map is not None:\n            model = model.to(device_map)\n        return model\n''')

with open("upr/metrics.py", "w", encoding="utf-8") as f:
    f.write('''import torch\nimport numpy as np\nfrom typing import Dict, Any\n\ndef compute_weight_metrics(original: torch.Tensor, reconstructed: torch.Tensor) -> Dict[str, Any]:\n    orig_f32 = original.detach().cpu().to(torch.float32)\n    recon_f32 = reconstructed.detach().cpu().to(torch.float32)\n    is_exact = bool(torch.equal(original.detach().cpu(), reconstructed.detach().cpu()))\n    diff = torch.abs(orig_f32 - recon_f32)\n    mae = float(diff.mean().item())\n    rmse = float(torch.sqrt(torch.mean((orig_f32 - recon_f32) ** 2)).item())\n    max_error = float(diff.max().item())\n    orig_flat = orig_f32.view(-1)\n    recon_flat = recon_f32.view(-1)\n    norm_orig = torch.norm(orig_flat)\n    norm_recon = torch.norm(recon_flat)\n    if norm_orig == 0 or norm_recon == 0:\n        cos_sim = 1.0 if norm_orig == norm_recon else 0.0\n    else:\n        cos_sim = float((torch.dot(orig_flat, recon_flat) / (norm_orig * norm_recon)).item())\n    return {"exact_match": is_exact, "mae": mae, "rmse": rmse, "max_error": max_error, "cosine_similarity": cos_sim, "num_elements": int(orig_f32.numel())}\n''')

from huggingface_hub import login
login(token=HF_TOKEN)

import numpy as np
import upr
print(f'UPR active. PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully.
Active Working Directory: /content/drive/MyDrive/UniversalPrecisionRuntime


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


UPR active. PyTorch: 2.11.0+cu128, CUDA available: True


### Step 2: Evaluate Original Baseline Model
Run forward passes and generation on the original FP16 Hugging Face model, then free memory.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen3.5-0.8B'
BITPLANE_DIR = 'models/bitplane_qwen'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
prompt = "Universal Precision Runtime allows dynamic precision reconstruction from a single bit-plane checkpoint."
inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)

print(f'Loading original FP16 baseline model on {DEVICE}...')
orig_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=torch.float16
).to(DEVICE)
orig_model.eval()

# Forward hook activations
orig_activations = {}
def get_hook(act_dict, layer_idx):
    def hook(module, input, output):
        hidden_states = output[0] if isinstance(output, tuple) else output
        act_dict[layer_idx] = hidden_states.detach().cpu().to(torch.float32)
    return hook

def get_layers(model):
    if hasattr(model, 'model') and hasattr(model.model, 'layers'):
        return model.model.layers
    elif hasattr(model, 'transformer') and hasattr(model.transformer, 'h'):
        return model.transformer.h
    else:
        raise AttributeError('Could not auto-detect Transformer layer list.')

hooks = []
for idx, layer in enumerate(get_layers(orig_model)):
    hooks.append(layer.register_forward_hook(get_hook(orig_activations, idx)))

with torch.no_grad():
    orig_logits = orig_model(**inputs).logits.detach().cpu().to(torch.float32)
    gen_prompt = "The key advantage of a single bit-plane representation is"
    gen_inputs = tokenizer(gen_prompt, return_tensors='pt').to(DEVICE)
    orig_output_ids = orig_model.generate(**gen_inputs, max_new_tokens=40, do_sample=False).detach().cpu()

for h in hooks:
    h.remove()

orig_text = tokenizer.decode(orig_output_ids[0], skip_special_tokens=True)
print(f'Original Baseline evaluation completed. Generated Text:\n"{orig_text}"')

# Clear memory to prevent OOM crash
del orig_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Cleared baseline model from VRAM.')

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading original FP16 baseline model on cuda...


[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Original Baseline evaluation completed. Generated Text:
"The key advantage of a single bit-plane representation is that it allows for the efficient representation of the entire data stream. This is because the bit-plane representation is a linear combination of the bit-plane representation of the data stream.

The bit-plane representation is"
Cleared baseline model from VRAM.


### Step 3: Evaluate 16-Bit Reconstructed BitPlane Model
Load the 16-bit reconstructed BitPlane model, run identical forward passes, and collect activations & logits.

In [3]:
print(f'Reconstructing 16-bit BitPlane model from {BITPLANE_DIR} on {DEVICE}...')
recon_model = upr.BitPlaneModel.from_pretrained(
    bitplane_directory=BITPLANE_DIR,
    bits=16,
    base_model_id=MODEL_ID,
    torch_dtype=torch.float16
).to(DEVICE)
recon_model.eval()

recon_activations = {}
hooks = []
for idx, layer in enumerate(get_layers(recon_model)):
    hooks.append(layer.register_forward_hook(get_hook(recon_activations, idx)))

with torch.no_grad():
    recon_logits = recon_model(**inputs).logits.detach().cpu().to(torch.float32)
    recon_output_ids = recon_model.generate(**gen_inputs, max_new_tokens=40, do_sample=False).detach().cpu()

for h in hooks:
    h.remove()

recon_text = tokenizer.decode(recon_output_ids[0], skip_special_tokens=True)
print(f'Reconstructed 16-Bit evaluation completed. Generated Text:\n"{recon_text}"')

# Clear reconstructed model memory
del recon_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Cleared reconstructed model from VRAM.')

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Reconstructing 16-bit BitPlane model from models/bitplane_qwen on cuda...
Instantiating model base architecture 'Qwen/Qwen3.5-0.8B' for precision bits=16...


Reconstructing (16-bit): 100%|██████████| 321/321 [01:17<00:00,  4.15it/s]


Reconstructed 16-Bit evaluation completed. Generated Text:
"The key advantage of a single bit-plane representation is that it allows for the efficient representation of the entire data stream. This is because the bit-plane representation is a linear combination of the bit-plane representation of the data stream.

The bit-plane representation is"
Cleared reconstructed model from VRAM.


### Step 4: Level 2 — Layer Activation Output Comparison
Compare layer-by-layer activation output tensors.

In [4]:
layer_metrics = {}
print(f"{'Layer':<10} | {'L2 Error':<15} | {'Cosine Sim':<15} | {'Max Difference':<15}")
print("-" * 65)

for idx in sorted(orig_activations.keys()):
    act_orig = orig_activations[idx]
    act_recon = recon_activations[idx]

    l2_err = float(torch.norm(act_orig - act_recon).item())
    max_diff = float(torch.max(torch.abs(act_orig - act_recon)).item())

    flat_o = act_orig.view(-1)
    flat_r = act_recon.view(-1)
    cos_sim = float((torch.dot(flat_o, flat_r) / (torch.norm(flat_o) * torch.norm(flat_r))).item())

    layer_metrics[f'layer_{idx}'] = {
        'l2_error': l2_err,
        'cosine_similarity': cos_sim,
        'max_difference': max_diff
    }
    print(f"{idx:<10} | {l2_err:<15.6e} | {cos_sim:<15.6f} | {max_diff:<15.6e}")

Layer      | L2 Error        | Cosine Sim      | Max Difference 
-----------------------------------------------------------------
0          | 0.000000e+00    | 1.000000        | 0.000000e+00   
1          | 0.000000e+00    | 1.000000        | 0.000000e+00   
2          | 0.000000e+00    | 1.000000        | 0.000000e+00   
3          | 0.000000e+00    | 1.000000        | 0.000000e+00   
4          | 0.000000e+00    | 1.000000        | 0.000000e+00   
5          | 0.000000e+00    | 1.000000        | 0.000000e+00   
6          | 0.000000e+00    | 1.000000        | 0.000000e+00   
7          | 0.000000e+00    | 1.000000        | 0.000000e+00   
8          | 0.000000e+00    | 1.000000        | 0.000000e+00   
9          | 0.000000e+00    | 1.000000        | 0.000000e+00   
10         | 0.000000e+00    | 1.000000        | 0.000000e+00   
11         | 0.000000e+00    | 1.000000        | 0.000000e+00   
12         | 0.000000e+00    | 1.000000        | 0.000000e+00   
13         | 0.000000e+0

### Step 5: Level 3 — Logit & Generation Comparison
Compute logit MAE, RMSE, Cosine similarity, KL Divergence, and Top-1 token agreement.

In [5]:
import torch.nn.functional as F

# Logit metrics
diff = torch.abs(orig_logits - recon_logits)
mae = float(diff.mean().item())
rmse = float(torch.sqrt(torch.mean((orig_logits - recon_logits)**2)).item())

flat_o = orig_logits.view(-1)
flat_r = recon_logits.view(-1)
cos_sim = float((torch.dot(flat_o, flat_r) / (torch.norm(flat_o) * torch.norm(flat_r))).item())

# KL Divergence
p_log_prob = F.log_softmax(recon_logits, dim=-1)
q_prob = F.softmax(orig_logits, dim=-1)
kl_div = float(F.kl_div(p_log_prob, q_prob, reduction='batchmean').item())

# Generation Token Agreement
top1_match = bool(torch.equal(orig_output_ids, recon_output_ids))
token_matches = (orig_output_ids == recon_output_ids).sum().item()
total_gen_tokens = orig_output_ids.numel()
token_acc = (token_matches / total_gen_tokens) * 100

logit_metrics = {
    'mae': mae,
    'rmse': rmse,
    'cosine_similarity': cos_sim,
    'kl_divergence': kl_div
}

print('='*60)
print('LEVEL 3 LOGIT & GENERATION RESULTS')
print(f'Logit MAE: {mae:.6e}')
print(f'Logit RMSE: {rmse:.6e}')
print(f'Logit Cosine Similarity: {cos_sim:.6f}')
print(f'KL Divergence: {kl_div:.6e}')
print('-'*60)
print(f'Top-1 Token Agreement: {token_acc:.2f}% ({token_matches}/{total_gen_tokens} tokens)')
print(f'Text Exact Match: {orig_text == recon_text}')
print('='*60)

LEVEL 3 LOGIT & GENERATION RESULTS
Logit MAE: 0.000000e+00
Logit RMSE: 0.000000e+00
Logit Cosine Similarity: 1.000098
KL Divergence: -7.246251e-08
------------------------------------------------------------
Top-1 Token Agreement: 100.00% (50/50 tokens)
Text Exact Match: True


### Step 6: Export Results
Save Level 2 and Level 3 metrics to `results/layer_outputs.json` and `results/logits_level2_3.json`.

In [6]:
import json

os.makedirs('results', exist_ok=True)

with open('results/layer_outputs.json', 'w') as f:
    json.dump(layer_metrics, f, indent=2)

level3_results = {
    'logit_metrics': logit_metrics,
    'generation_metrics': {
        'top1_match': top1_match,
        'token_accuracy_pct': token_acc,
        'orig_text': orig_text,
        'recon_text': recon_text
    }
}

with open('results/logits_level2_3.json', 'w') as f:
    json.dump(level3_results, f, indent=2)

print('Successfully saved Level 2 & Level 3 evaluation artifacts to results/ directory!')

Successfully saved Level 2 & Level 3 evaluation artifacts to results/ directory!
